# Dense-grid calibration: three reference Stokes maps

Constructs and compares three physically meaningful references from the four dense-grid GRASP simulations:

- **`x_off`**: no-grid telescope, x-polarized input
- **`y_off`**: no-grid telescope, y-polarized input
- **`x_on`**: dense-grid-on, x-polarized input
- **`y_on`**: dense-grid-on, y-polarized input

| Reference | Formula | Physical question |
|-----------|---------|-------------------|
| **A** | `S_unpol_off = 0.5*(S_x_off + S_y_off)` | Baseline instrumental polarization of the bare telescope? |
| **B** | `S_unpol_on = 0.5*(S_x_on + S_y_on)` | What does the actual grid calibrator produce from unpolarized input? |
| **C** | `S_ideal = 0.5*S_y_off` | What should the calibrator look like if the grid only selects y-pol without changing the optics? |

The comparison B vs C is the core diagnostic: if they agree, the dense grid acts as an ideal polarization selector and the calibration transfers to the grid-off telescope.

> **Physical note:** Stokes parameters are combined incoherently — complex fields are **not** summed.
> The factor of 1/2 represents equal x and y power in unpolarized input.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
# ---------------------------------------------------------------
# Edit these paths to point to your GRASP .grd output files.
# All four files must share the same coordinate grid.
# ---------------------------------------------------------------
data_dir = "../data/wire_on_off/"

file_xoff = data_dir + "azel_1deg_offx.grd"    # no-grid, x input polarization
file_yoff = data_dir + "azel_1deg_offy.grd"    # no-grid, y input polarization
file_xon  = data_dir + "azel_1deg_densex.grd"  # dense-grid-on, x input polarization
file_yon  = data_dir + "azel_1deg_densey.grd"  # dense-grid-on, y input polarization


## 1. Load four dense-grid runs


In [ ]:
def load_grasp_grd(filepath):
    # Load GRASP .grd file. Returns x, y (1D), F1, F2 (2D complex, shape ny x nx).
    with open(filepath, 'r') as f:
        lines = f.readlines()
    grid_idx = None
    for i, line in enumerate(lines):
        parts = line.strip().split()
        if len(parts) == 3:
            try:
                nx, ny = int(parts[0]), int(parts[1])
                grid_idx = i
                break
            except Exception:
                continue
    if grid_idx is None:
        raise ValueError(f"Could not find grid size line in {filepath}")
    data = np.loadtxt(lines[grid_idx + 1:])
    F1 = (data[:, 0] + 1j * data[:, 1]).reshape(ny, nx)
    F2 = (data[:, 2] + 1j * data[:, 3]).reshape(ny, nx)
    for i in range(grid_idx):
        parts = lines[i].strip().split()
        if len(parts) == 4:
            try:
                xmin, ymin, xmax, ymax = map(float, parts)
            except Exception:
                continue
    x = np.linspace(xmin, xmax, nx)
    y = np.linspace(ymin, ymax, ny)
    return x, y, F1, F2


def compute_stokes(F1, F2):
    # Stokes I, Q, U, V from complex field components (linear basis).
    E1E1  = np.abs(F1)**2
    E2E2  = np.abs(F2)**2
    cross = F1 * np.conj(F2)
    I = E1E1 + E2E2
    Q = E1E1 - E2E2
    U = 2.0 * np.real(cross)
    V = -2.0 * np.imag(cross)
    return I, Q, U, V


In [ ]:
def make_db_mask(I, db_min=-20):
    # Boolean mask: True where I >= db_min dB relative to peak.
    I_db = 10.0 * np.log10(np.maximum(I, 1e-300) / np.nanmax(I))
    return I_db >= db_min


def pol_fraction(Q, U, I):
    # Linear polarization fraction p = sqrt(Q^2+U^2)/I.
    return np.sqrt(Q**2 + U**2) / np.maximum(I, 1e-30)


def pol_angle_deg(Q, U):
    # Linear polarization angle psi = 0.5*atan2(U, Q) in degrees.
    return 0.5 * np.degrees(np.arctan2(U, Q))


def pol_angle_diff_deg(psi_a, psi_b):
    # Wrapped difference psi_a - psi_b in [-90, 90) deg.
    # Uses doubled-angle space to handle the 180-deg ambiguity correctly.
    two_a = np.radians(2.0 * psi_a)
    two_b = np.radians(2.0 * psi_b)
    return np.degrees(0.5 * np.arctan2(np.sin(two_a - two_b), np.cos(two_a - two_b)))


def plot_stokes_row(x, y, I, Q, U, title="", db_mask_threshold=-40):
    # 5-panel row: I(dB), Q/I, U/I, pol fraction p, pol angle psi.
    # Polarization maps are masked where I < db_mask_threshold dB.
    I_db  = 10 * np.log10(np.maximum(I, 1e-300) / np.nanmax(I))
    lo    = I_db < db_mask_threshold
    p     = pol_fraction(Q, U, I)
    psi   = pol_angle_deg(Q, U)
    QI    = np.where(lo, np.nan, Q / np.maximum(I, 1e-30))
    UI    = np.where(lo, np.nan, U / np.maximum(I, 1e-30))
    p_m   = np.where(lo, np.nan, p)
    psi_m = np.where(lo, np.nan, psi)
    ext   = [x.min(), x.max(), y.min(), y.max()]
    fig, axs = plt.subplots(1, 5, figsize=(22, 4))
    kw = dict(extent=ext, origin="lower", aspect="equal")
    im = axs[0].imshow(I_db, cmap="viridis", vmin=-50, vmax=0, **kw)
    axs[0].set_title("I [dB re peak]")
    plt.colorbar(im, ax=axs[0], label="dB")
    for ax, d, lab in [(axs[1], QI, "Q / I"), (axs[2], UI, "U / I")]:
        fin = np.abs(d[np.isfinite(d)])
        lim = float(fin.max()) if fin.size > 0 else 1.0
        im = ax.imshow(d, cmap="coolwarm", vmin=-lim, vmax=lim, **kw)
        ax.set_title(lab)
        plt.colorbar(im, ax=ax)
    im = axs[3].imshow(p_m, cmap="plasma", vmin=0, vmax=1, **kw)
    axs[3].set_title("p = sqrt(Q²+U²)/I")
    plt.colorbar(im, ax=axs[3])
    im = axs[4].imshow(psi_m, cmap="twilight", vmin=-90, vmax=90, **kw)
    axs[4].set_title("ψ [deg]")
    plt.colorbar(im, ax=axs[4], label="deg")
    for ax in axs:
        ax.set_xlabel("x [deg]")
        ax.set_ylabel("y [deg]")
    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    plt.show()


def plot_sym_map(x, y, data, title="", cbar_label="", cmap="coolwarm",
                 percentile=99.5, mask=None):
    # Symmetric (zero-centered) colorbar map. mask: True = include pixel.
    d   = np.where(mask, data, np.nan) if mask is not None else np.array(data, dtype=float)
    fin = d[np.isfinite(d)]
    if fin.size == 0:
        print(f"No finite pixels: {title}")
        return
    vmax = float(np.nanpercentile(np.abs(fin), percentile))
    ext  = [x.min(), x.max(), y.min(), y.max()]
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(d, extent=ext, origin="lower", aspect="equal",
                   cmap=cmap, vmin=-vmax, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel("x [deg]")
    ax.set_ylabel("y [deg]")
    plt.colorbar(im, ax=ax, label=cbar_label)
    plt.tight_layout()
    plt.show()


def summarize_pol(I, Q, U, label="", db_cut=-20):
    # Print beam-weighted pol fraction and effective integrated pol angle.
    mask = make_db_mask(I, db_min=db_cut)
    p    = pol_fraction(Q, U, I)
    w    = I * mask.astype(float)
    wsum = w.sum()
    print(f"\n--- {label} (I >= {db_cut} dB, {mask.sum()} pixels) ---")
    if wsum <= 0:
        print("  No valid pixels.")
        return
    # Beam-weighted mean pol fraction: sum(sqrt(Q^2+U^2)) / sum(I) in mask.
    wp_mean = np.sum(w * p) / wsum
    # Effective pol angle: angle of the integrated Stokes vector (Ren-style).
    # From ren_effective_angle_analytic: phi = 0.5 * atan2(sum(U), sum(Q)).
    # Note: w = I*mask, so np.sum(w*Q)/np.sum(w*I) would be I^2-weighted — wrong.
    # The correct formula sums Q and U directly over the mask pixels.
    Q_int   = float(np.sum(Q[mask]))
    U_int   = float(np.sum(U[mask]))
    psi_eff = 0.5 * np.degrees(np.arctan2(U_int, Q_int))
    print(f"  beam-weighted mean p      = {wp_mean:.6f}")
    print(f"  effective pol angle psi   = {psi_eff:.4f} deg")
    print(f"  total I in region         = {np.sum(I[mask]):.6e}")


In [ ]:
print("Loading files...")
x_xoff, y_xoff, F1_xoff, F2_xoff = load_grasp_grd(file_xoff)
x_yoff, y_yoff, F1_yoff, F2_yoff = load_grasp_grd(file_yoff)
x_xon,  y_xon,  F1_xon,  F2_xon  = load_grasp_grd(file_xon)
x_yon,  y_yon,  F1_yon,  F2_yon  = load_grasp_grd(file_yon)

# Verify all grids are identical
for name, xi, yi in [("y_off", x_yoff, y_yoff),
                     ("x_on",  x_xon,  y_xon),
                     ("y_on",  x_yon,  y_yon)]:
    assert np.allclose(xi, x_xoff), f"x-grid mismatch: {name}"
    assert np.allclose(yi, y_xoff), f"y-grid mismatch: {name}"

x, y = x_xoff, y_xoff
print(f"Grid shape : {F1_xoff.shape}")
print(f"x: {x.min():.3f} to {x.max():.3f} deg  ({len(x)} points)")
print(f"y: {y.min():.3f} to {y.max():.3f} deg  ({len(y)} points)")
print("All four grids match: OK")

# Compute Stokes for each run
Ix,   Qx,   Ux,   _ = compute_stokes(F1_xoff, F2_xoff)  # x_off
Iy,   Qy,   Uy,   _ = compute_stokes(F1_yoff, F2_yoff)  # y_off
Ixon, Qxon, Uxon, _ = compute_stokes(F1_xon,  F2_xon)   # x_on
Iyon, Qyon, Uyon, _ = compute_stokes(F1_yon,  F2_yon)   # y_on

print("\nPeak intensity for each run:")
for lbl, I in [("x_off", Ix), ("y_off", Iy), ("x_on", Ixon), ("y_on", Iyon)]:
    print(f"  {lbl:8s}: I_peak = {np.nanmax(I):.6e}")


## 2. Reference A: no-grid unpolarized response

$$S_{\rm unpol\_off} = \frac{1}{2}\bigl(S_{x\_off} + S_{y\_off}\bigr)$$

Simulates unpolarized input through the bare telescope with no grid.
The resulting $p$ and $\psi$ maps show **baseline instrumental polarization** of the telescope alone.

- If $p_A \approx 0$ in the main beam: the telescope is a clean system before any grid is introduced.
- If $p_A$ is non-negligible: instrumental polarization is present regardless of the grid.

This is **not** a calibration signal; it is the floor against which the grid-induced calibration has to be assessed.


In [ ]:
# Reference A: no-grid unpolarized
# Stokes parameters added incoherently; complex fields are NOT combined.
IA = 0.5 * (Ix + Iy)
QA = 0.5 * (Qx + Qy)
UA = 0.5 * (Ux + Uy)

plot_stokes_row(x, y, IA, QA, UA,
                title="Reference A: No-grid unpolarized  (0.5 x_off + 0.5 y_off)")
summarize_pol(IA, QA, UA, label="Reference A: no-grid unpolarized", db_cut=-20)


## 3. Reference B: actual dense-grid response to unpolarized input

$$S_{\rm unpol\_on} = \frac{1}{2}\bigl(S_{x\_on} + S_{y\_on}\bigr)$$

Simulates the dense-grid + telescope system as seen by an unpolarized warm source.
Because the dense grid strongly suppresses x-pol and largely transmits y-pol,
this is expected to be predominantly y-like ($Q/I \approx -1$, $p \approx 1$).

This is the **actual simulated calibration signal** from the dense grid.


In [ ]:
# Reference B: actual dense-grid response to unpolarized input
IB = 0.5 * (Ixon + Iyon)
QB = 0.5 * (Qxon + Qyon)
UB = 0.5 * (Uxon + Uyon)

plot_stokes_row(x, y, IB, QB, UB,
                title="Reference B: Dense-grid unpolarized  (0.5 x_on + 0.5 y_on)")
summarize_pol(IB, QB, UB, label="Reference B: dense-grid unpolarized", db_cut=-20)


## 4. Reference C: ideal y-selector through the no-grid telescope

$$S_{\rm ideal} = \frac{1}{2} S_{y\_off}$$

An **ideal** wire grid would:
1. transmit y-polarization completely, without modification,
2. block x-polarization entirely,
3. leave the telescope optics otherwise unchanged.

For an unpolarized source (equal x and y power), the ideal grid selects the y component only,
giving half the y-pol no-grid response.

**This is the reference for "what should the calibrator look like if the grid only selects y-pol?"**

If Reference B $\approx$ Reference C, the dense grid acts as an ideal selector and the calibration
transfers to the normal grid-off telescope.
If they differ, the grid perturbs the optical response and the calibration would
characterize the "LAT with grid" rather than the normal telescope.


In [ ]:
# Reference C: ideal y-selector through the no-grid telescope
IC = 0.5 * Iy
QC = 0.5 * Qy
UC = 0.5 * Uy

plot_stokes_row(x, y, IC, QC, UC,
                title="Reference C: Ideal y-selector  (0.5 y_off)")
summarize_pol(IC, QC, UC, label="Reference C: ideal y-selector", db_cut=-20)


## 5. Compare the three references

### What each comparison shows:

**Polarization fraction** $p_A$, $p_B$, $p_C$:
- $p_A$: baseline instrumental polarization of the bare telescope (should be small).
- $p_B$: actual polarization produced by the grid calibrator.
- $p_C$: polarization from the ideal selector (~1 in the main beam, since only y is transmitted).

**Grid error angle** $\psi_B - \psi_C$:
- The most important test. Small and structureless $\Rightarrow$ grid is an ideal selector.
- Structured $\Rightarrow$ grid modifies the polarized beam beyond simple y-transmission.

**Normalized Stokes residuals** $(Q_B - Q_C)/I_C$, $(U_B - U_C)/I_C$, $(I_B - I_C)/I_C$:
- Local departure of the actual grid-on Stokes from the ideal selector, in units of $I_{\rm ideal}$.

> Polarization angle maps are unreliable where $p$ is very small.  
> Always interpret $\psi_B - \psi_C$ alongside $p$ and $I$, and apply a dB mask.


In [ ]:
# Polarization fraction comparison: A, B, C
# dB mask referenced to IC (ideal selector).
mask20 = make_db_mask(IC, db_min=-20)

pA = np.where(mask20, pol_fraction(QA, UA, IA), np.nan)
pB = np.where(mask20, pol_fraction(QB, UB, IB), np.nan)
pC = np.where(mask20, pol_fraction(QC, UC, IC), np.nan)

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
ext = [x.min(), x.max(), y.min(), y.max()]
kw  = dict(extent=ext, origin="lower", aspect="equal", cmap="plasma", vmin=0, vmax=1)

for ax, d, lbl in [(axs[0], pA, "A: no-grid unpol"),
                   (axs[1], pB, "B: grid-on unpol"),
                   (axs[2], pC, "C: ideal y-sel")]:
    im = ax.imshow(d, **kw)
    ax.set_title(f"p  ({lbl})")
    ax.set_xlabel("x [deg]")
    ax.set_ylabel("y [deg]")
    plt.colorbar(im, ax=ax, label="p")

plt.suptitle("Polarization fraction above -20 dB  (dB mask referenced to I_C)", fontsize=11)
plt.tight_layout()
plt.show()

# Beam-weighted mean pol fraction
print("Beam-weighted mean pol fraction above -20 dB (referenced to I_C):")
for lbl, I, Q, U in [("A (no-grid unpol)", IA, QA, UA),
                     ("B (grid-on unpol)", IB, QB, UB),
                     ("C (ideal y-sel)",   IC, QC, UC)]:
    w    = I * mask20
    wsum = w.sum()
    p    = pol_fraction(Q, U, I)
    wp   = np.sum(w * p) / wsum if wsum > 0 else np.nan
    print(f"  Reference {lbl}: {wp:.6f}")


In [ ]:
# Grid error angle: psi_B - psi_C (wrapped to [-90, 90))
psi_B = pol_angle_deg(QB, UB)
psi_C = pol_angle_deg(QC, UC)

# Mask: -20 dB AND pol fraction > 2%  (angle maps unreliable where p is tiny)
p_min    = 0.02
mask_ang = mask20 & (pol_fraction(QB, UB, IB) > p_min) & (pol_fraction(QC, UC, IC) > p_min)

delta_psi = pol_angle_diff_deg(psi_B, psi_C)
delta_psi_masked = np.where(mask_ang, delta_psi, np.nan)

plot_sym_map(x, y, delta_psi_masked,
             title="Grid error angle  psi_B - psi_C  (masked: -20 dB, p > 2%)",
             cbar_label="delta-psi [deg]", percentile=99.5)

# Stokes residuals normalized by I_C
dQ_norm = np.where(mask20, (QB - QC) / np.maximum(IC, 1e-30), np.nan)
dU_norm = np.where(mask20, (UB - UC) / np.maximum(IC, 1e-30), np.nan)
dI_norm = np.where(mask20, (IB - IC) / np.maximum(IC, 1e-30), np.nan)

for d, lbl in [(dQ_norm, "(Q_B - Q_C) / I_C"),
               (dU_norm, "(U_B - U_C) / I_C"),
               (dI_norm, "(I_B - I_C) / I_C")]:
    plot_sym_map(x, y, d, title=lbl, cbar_label="fraction", percentile=99.5)


In [ ]:
# Weighted summary table for the grid error angle over several dB thresholds.
# Reference: I_C (ideal selector).  Weight: I_B (actual calibrator intensity).

rows = []
for db_cut in [-10, -20, -30, -40, -50]:
    mask = make_db_mask(IC, db_min=db_cut) & np.isfinite(delta_psi)
    if not np.any(mask):
        continue
    d    = delta_psi[mask]
    w    = np.maximum(IB[mask], 0.0)
    wsum = w.sum()
    if wsum <= 0:
        continue
    rows.append({
        "dB cut":             db_cut,
        "pixels":             int(mask.sum()),
        "w.mean dpsi [deg]": float(np.sum(w * d) / wsum),
        "w.|dpsi| [deg]":    float(np.sum(w * np.abs(d)) / wsum),
        "w.RMS dpsi [deg]":  float(np.sqrt(np.sum(w * d**2) / wsum)),
        "max |dpsi| [deg]":  float(np.max(np.abs(d))),
    })

df = pd.DataFrame(rows)
print("Grid error angle psi_B - psi_C  (weight = I_actual, dB ref = I_ideal):")
display(df)


### Interpretation

**Reference A** diagnoses the baseline instrumental polarization of the telescope alone.
A small $p_A$ confirms the telescope is well-behaved before any grid is added.
This is not a calibration signal; it is the background floor.

**Reference B** is the actual simulated calibration signal: what the real instrument measures
from an unpolarized source with the dense grid in place.
The large $p_B$ confirms the grid successfully produces a strongly polarized output.

**Reference C** is the ideal target: what the calibrator should produce if the grid only selects
y-polarization without changing the telescope optics.

**The comparison B vs C** is the core test:
- Small $\psi_B - \psi_C$ and small Stokes residuals $\Rightarrow$ the grid is an ideal selector;
  calibration transfers to the grid-off telescope.
- Structured residuals $\Rightarrow$ the grid modifies local polarized beam structure;
  the calibrator characterizes "LAT with grid" rather than the normal telescope.

See `dense_grid_cleanest_test.ipynb` for the focused quantitative test with full beam-weighted statistics.
